In [ ]:
import os
import numpy as np  # Импортируем библиотеку NumPy для работы с массивами чисел
import pandas as pd  # Импортируем Pandas для загрузки и анализа табличных данных
import torch  # Импортируем сам фреймворк PyTorch для построения нейросетей
import torch.nn as nn  # Импортируем модуль готовых слоев нейросетей (Linear, и т.д.)
import torch.optim as optim  # Импортируем модуль с алгоритмами оптимизации (SGD, Adam)
from sklearn.metrics import (
    r2_score,
)  # Импортируем функцию расчета метрики R2 из scikit-learn
from sklearn.model_selection import (
    train_test_split,
)  # Импортируем функцию разделения выборки на train/test
from sklearn.preprocessing import (
    StandardScaler,
)  # Импортируем инструмент для стандартизации (масштабирования) признаков

# Загружаем датасет из CSV-файла в таблицу Pandas DataFrame
current_dir = os.path.dirname(os.path.abspath('__file__'))
path_to_file = os.path.join(current_dir, 'Fish.csv')
df = pd.read_csv(path_to_file)

# Создаем матрицу признаков X, удаляя из исходной таблицы целевую колонку Weight
X = df.drop(columns=["Weight"])

# Выделяем целевую переменную y (вес рыбы), переводим в массив NumPy и тип float32
y = df["Weight"].values.astype(np.float32)

# Преобразуем текстовую колонку Species в бинарные признаки (0 и 1) через One-Hot Encoding
X = pd.get_dummies(X, columns=["Species"], drop_first=True)

# Переводим готовую таблицу признаков в чистый массив NumPy с типом данных float32
X = X.values.astype(np.float32)

# Делим данные: 80% на обучение, 20% на тест. Задаем random_state для воспроизводимости
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Создаем объект StandardScaler для приведения признаков к среднему 0 и отклонению 1
scaler = StandardScaler()

# На основе обучающей выборки считаем среднее/std и сразу же масштабируем её
X_train = scaler.fit_transform(X_train)

# Применяем те же самые параметры масштабирования (среднее и std) к тестовой выборке
X_test = scaler.transform(X_test)

# Конвертируем обучающие признаки из массива NumPy в тензор PyTorch
X_train_t = torch.tensor(X_train)

# Конвертируем обучающие ответы в тензор и добавляем размерность (из [N] делаем [N, 1])
y_train_t = torch.tensor(y_train).unsqueeze(1)

# Конвертируем тестовые признаки из массива NumPy в тензор PyTorch
X_test_t = torch.tensor(X_test)

# Конвертируем тестовые ответы в тензор и тоже добавляем размерность до [N, 1]
y_test_t = torch.tensor(y_test).unsqueeze(1)

# Считаем количество входных признаков (количество колонок в нашей матрице X)
input_dim = X_train.shape[1]

# Создаем однослойный персептрон: линейный слой с input_dim входами и 1 выходом
model = nn.Linear(input_dim, 1)

# Задаем функцию потерь MSE (Mean Squared Error) для задачи регрессии
criterion = nn.MSELoss()

# Инициализируем оптимизатор SGD, передаем ему веса модели и скорость обучения (lr)
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Задаем количество полных проходов по обучающему датасету (эпох)
epochs = 2000

# Запускаем цикл обучения
for epoch in range(epochs):
    # Переводим модель в режим обучения (включает вычисление градиентов)
    model.train()

    # Делаем предсказание: пропускаем признаки через веса линейного слоя (Forward pass)
    predictions = model(X_train_t)

    # Вычисляем величину ошибки (MSE) между предсказаниями и реальными весами рыб
    loss = criterion(predictions, y_train_t)

    # Очищаем градиенты прошлых шагов, чтобы они не накапливались
    optimizer.zero_grad()

    # Считаем градиенты ошибки по всем весам модели (Backward pass)
    loss.backward()

    # Делаем шаг оптимизатора: обновляем веса модели в сторону уменьшения ошибки
    optimizer.step()

    # Каждые 400 эпох выводим текущий прогресс на экран
    if (epoch + 1) % 400 == 0:
        # Печатаем номер эпохи и текущее значение функции потерь
        print(f"Эпоха [{epoch+1}/{epochs}], Ошибка (MSE): {loss.item():.4f}")

# Переводим модель в режим оценки (выключает специфичные для обучения слои)
model.eval()

# Отключаем автоматическое отслеживание и расчет градиентов для экономии памяти
with torch.no_grad():
    # Пропускаем тестовые признаки через обученную модель для получения прогнозов
    y_pred_t = model(X_test_t)

    # Переводим тензор предсказаний обратно в обычный массив NumPy
    y_pred = y_pred_t.numpy()

# Рассчитываем итоговую метрику R2 (коэффициент детерминации) на тесте
r2 = r2_score(y_test, y_pred)

# Выводим финальный результат работы нашей нейросети
print(f"\nРезультат R2 на тестовой выборке: {r2:.4f}")
